### Procesamiento de Lenguaje Natural I

# **Desafío 1**




In [1]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.naive_bayes import MultinomialNB, ComplementNB

from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups

import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.



Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.



Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))

print(f'shape: {X_train.shape}')

print(f'Cantidad de documentos: {X_train.shape[0]}')

print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.



El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target

y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')

newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811

print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]

print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:

  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()

clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos

del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)

y_test = newsgroups_test.target

y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.



* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.

* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**





**1. Vectorizar documentos**

* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.

Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido

la similaridad según el contenido del texto y la etiqueta de clasificación.



**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.



**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**



* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros

de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.



**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.



**4. Transponer la matriz documento-término.**

* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.

* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.



**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Parte 1 - Similaridad entre documentos

Tomamos 5 documentos al azar del conjunto de entrenamiento (ya vectorizados en `X_train` con el `tfidfvect` definido más arriba) y para cada uno buscamos los 5 documentos más similares según similaridad coseno.

In [24]:
np.random.seed(42)
sample_idxs = np.random.choice(X_train.shape[0], size=5, replace=False)
print("Índices de documentos seleccionados:", sample_idxs)

for idx in sample_idxs:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    # excluimos la posición 0 porque siempre es el propio documento (similaridad = 1)
    mostsim = np.argsort(cossim)[::-1][1:6]

    print("=" * 90)
    print(f"Documento base (idx={idx}) | Clase real: {newsgroups_train.target_names[y_train[idx]]}")
    print("-" * 90)
    print(newsgroups_train.data[idx][:300].replace("\n", " "), "...")
    print()
    print("Top 5 documentos más similares:")
    for rank, i in enumerate(mostsim, start=1):
        clase = newsgroups_train.target_names[y_train[i]]
        print(f"  {rank}. idx={i} | similitud coseno={cossim[i]:.4f} | clase={clase}")
        print("     ", newsgroups_train.data[i][:150].replace("\n", " "), "...")
    print()

Índices de documentos seleccionados: [7492 3546 5582 4793 3813]
Documento base (idx=7492) | Clase real: comp.sys.mac.hardware
------------------------------------------------------------------------------------------
Could someone please post any info on these systems.  Thanks. BoB --  ----------------------------------------------------------------------  Robert Novitskey | "Pursuing women is similar to banging one's head rrn@po.cwru.edu  |  against a wall...with less opportunity for reward"  ...

Top 5 documentos más similares:
  1. idx=10935 | similitud coseno=0.6665 | clase=comp.sys.mac.hardware
      Hey everybody:     I want to buy a mac and I want to get a good price...who doesn't?  So, could anyone out there who has found a really good deal on a ...
  2. idx=7258 | similitud coseno=0.3476 | clase=comp.sys.ibm.pc.hardware
      Hay all:      Has anyone out there heard of any performance stats on the fabled p24t.  I was wondering what it's performance compared to the 486/66 an ..

**Interpretación (Parte 1):**

La similaridad coseno con TF-IDF captura solapamiento de vocabulario, no significado
semántico profundo. En la corrida, el documento base con idx=7492 (clase
`comp.sys.mac.hardware`) tuvo 3 de sus 5 vecinos más similares en esa misma clase,
confirmando que documentos de una misma categoría comparten vocabulario técnico
específico. El segundo vecino más cercano perteneció a `comp.sys.ibm.pc.hardware` —
clase distinta pero temáticamente cercana (ambas hablan de hardware de PC) — lo que
muestra que la similaridad también agrupa por vocabulario compartido entre clases
afines, no solo por la etiqueta exacta. Esto confirma que la similaridad coseno sobre
TF-IDF es una buena aproximación léxica, aunque no distingue perfectamente entre
clases muy relacionadas ni captura sinónimos o contexto semántico.

## Parte 2 - Modelo de clasificación por prototipos (zero-shot)

Construimos un clasificador de tipo *zero-shot* / vecino más cercano (1-NN): para cada documento de test, lo
comparamos (similaridad coseno) contra **todos** los documentos de entrenamiento y le asignamos la clase del
documento de entrenamiento con mayor similaridad. No hay entrenamiento de un modelo propiamente dicho: la
"predicción" surge exclusivamente de la geometría de los vectores TF-IDF.

Usamos `pairwise_distances_argmin` con `metric='cosine'`, que internamente procesa los datos en bloques (chunks),
evitando construir en memoria la matriz completa de similaridad de 7532 x 11314 documentos.

In [25]:
from sklearn.metrics.pairwise import pairwise_distances_argmin

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

# Para cada documento de test, buscamos el índice del documento de train más similar (distancia coseno mínima)
nearest_train_idx = pairwise_distances_argmin(X_test, X_train, metric='cosine')

# Le asignamos la clase de ese documento de entrenamiento (modelo por prototipos / 1-NN)
y_pred_proto = y_train[nearest_train_idx]

f1_macro_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-score Macro (modelo por prototipos, 1-NN con similaridad coseno): {f1_macro_proto:.4f}")

F1-score Macro (modelo por prototipos, 1-NN con similaridad coseno): 0.5050


**Interpretación (Parte 2):**

Este enfoque es "zero-shot" porque no ajusta ningún parámetro a partir de los datos:
solo busca el vecino más parecido en el espacio TF-IDF y le copia la clase. El F1
macro obtenido fue **0.5050**, sensiblemente menor al mejor resultado de Naïve Bayes
(0.6954, Parte 3). Esto confirma que depender de un único documento de referencia es
más frágil que un modelo que agrega estadística de todos los documentos de una clase:
un solo vecino "raro" o ruidoso puede arrastrar una predicción equivocada, mientras
que Naïve Bayes promedia esa variabilidad al estimar probabilidades condicionadas
por clase usando todo el conjunto de entrenamiento.

## Parte 3 - Optimización de modelos Naïve Bayes

Probamos distintas configuraciones de vectorizador (`CountVectorizer` y `TfidfVectorizer`, variando `stop_words`,
`min_df`, `max_df` y `sublinear_tf`) combinadas con distintos modelos Naïve Bayes (`MultinomialNB` y `ComplementNB`)
y distintos valores de `alpha` (suavizado de Laplace/Lidstone).

**No se modifica `ngram_range`** en ninguna configuración (queda en su valor por defecto `(1, 1)`), tal como pide la
consigna. El objetivo es maximizar el F1-score Macro sobre el conjunto de test.

In [26]:
import pandas as pd

vectorizer_configs = [
    {"nombre": "TFIDF default",                         "vectorizer": TfidfVectorizer()},
    {"nombre": "TFIDF sublinear_tf",                     "vectorizer": TfidfVectorizer(sublinear_tf=True)},
    {"nombre": "TFIDF stopwords",                        "vectorizer": TfidfVectorizer(stop_words='english')},
    {"nombre": "TFIDF stopwords+sublinear+min_df+max_df","vectorizer": TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=2, max_df=0.9)},
    {"nombre": "Count default",                          "vectorizer": CountVectorizer()},
    {"nombre": "Count stopwords+min_df+max_df",           "vectorizer": CountVectorizer(stop_words='english', min_df=2, max_df=0.9)},
]

model_configs = [
    {"nombre": "MultinomialNB (alpha=1.0)",  "model": lambda: MultinomialNB(alpha=1.0)},
    {"nombre": "MultinomialNB (alpha=0.1)",  "model": lambda: MultinomialNB(alpha=0.1)},
    {"nombre": "MultinomialNB (alpha=0.01)", "model": lambda: MultinomialNB(alpha=0.01)},
    {"nombre": "ComplementNB (alpha=1.0)",   "model": lambda: ComplementNB(alpha=1.0)},
    {"nombre": "ComplementNB (alpha=0.1)",   "model": lambda: ComplementNB(alpha=0.1)},
    {"nombre": "ComplementNB (alpha=0.01)",  "model": lambda: ComplementNB(alpha=0.01)},
]

resultados = []

for vconf in vectorizer_configs:
    vect = vconf["vectorizer"]
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)

    for mconf in model_configs:
        modelo = mconf["model"]()
        modelo.fit(Xtr, y_train)
        y_pred = modelo.predict(Xte)
        f1 = f1_score(y_test, y_pred, average='macro')
        resultados.append({
            "vectorizador": vconf["nombre"],
            "modelo": mconf["nombre"],
            "f1_macro": f1,
        })

resultados_df = pd.DataFrame(resultados).sort_values("f1_macro", ascending=False).reset_index(drop=True)
resultados_df

,vectorizador,modelo,f1_macro
0,TFIDF default,ComplementNB (alpha=0.1),0.695365
1,TFIDF stopwords,ComplementNB (alpha=1.0),0.693611
2,TFIDF default,ComplementNB (alpha=1.0),0.692953
3,TFIDF sublinear_tf,ComplementNB (alpha=0.1),0.692682
4,TFIDF stopwords+sublinear+min_df+max_df,ComplementNB (alpha=1.0),0.692060
5,TFIDF sublinear_tf,ComplementNB (alpha=1.0),0.692043
6,TFIDF stopwords,ComplementNB (alpha=0.1),0.691919
7,TFIDF stopwords+sublinear+min_df+max_df,ComplementNB (alpha=0.1),0.687836
8,TFIDF stopwords,MultinomialNB (alpha=0.01),0.684439
9,TFIDF default,MultinomialNB (alpha=0.01),0.682861


In [27]:
mejor = resultados_df.iloc[0]
print("Mejor combinación encontrada:")
print(f"  Vectorizador: {mejor['vectorizador']}")
print(f"  Modelo:       {mejor['modelo']}")
print(f"  F1 Macro:     {mejor['f1_macro']:.4f}")

Mejor combinación encontrada:
  Vectorizador: TFIDF default
  Modelo:       ComplementNB (alpha=0.1)
  F1 Macro:     0.6954


**Interpretación (Parte 3):**

La mejor combinación encontrada fue **TFIDF default + ComplementNB (alpha=0.1)**,
con F1 macro = **0.6954**. Los mejores resultados generales se concentraron en
vectorizadores TF-IDF combinados con ComplementNB, aunque las diferencias entre las
4 variantes de TF-IDF probadas (default, stopwords, sublinear_tf, y la combinación
de las tres) fueron muy pequeñas (todas entre 0.687 y 0.695), por lo que el
preprocesamiento adicional no aportó una mejora clara sobre el TF-IDF por defecto.

El comportamiento de `alpha` fue distinto según el modelo: en `MultinomialNB`, bajar
`alpha` mejoró el desempeño de forma consistente y marcada (por ejemplo con TFIDF
default: 0.585 con alpha=1.0, 0.656 con alpha=0.1, 0.683 con alpha=0.01). En cambio,
en `ComplementNB` ocurrió lo contrario: `alpha=0.01` empeoró el resultado frente a
`alpha=0.1` y `alpha=1.0`, que se mantuvieron parejos entre sí. Como consecuencia,
con `alpha=0.01` `MultinomialNB` llegó a superar a `ComplementNB` en 3 de los 4
vectorizadores TF-IDF — la ventaja de `ComplementNB` no fue universal, sino que
dependió del nivel de suavizado usado.

Con `CountVectorizer` (conteos crudos, sin normalizar por TF-IDF) el desempeño fue
notablemente inferior en todos los casos (F1 entre 0.51 y 0.64), confirmando que
normalizar por frecuencia de documento (TF-IDF) es clave para este dataset.

Comparado contra el modelo por prototipos (F1=0.5050, Parte 2), la mejora del mejor
Naïve Bayes es sustancial (+0.19), mostrando el valor de aprender explícitamente la
distribución de palabras por clase en lugar de depender de un solo vecino.

## Parte 4 - Similaridad entre palabras (matriz término-documento)

Transponemos la matriz documento-término `X_train` (documentos x vocabulario) para obtener la matriz
término-documento `X_train_T` (vocabulario x documentos). Cada fila de `X_train_T` es ahora un vector que representa
una **palabra** en función de los documentos donde aparece (y con qué peso TF-IDF), y podemos medir similaridad
coseno entre palabras de la misma forma que hicimos entre documentos.

Elegimos manualmente 5 palabras interpretables y representativas de distintos temas del dataset, para evitar caer en
términos poco informativos (números sueltos, fragmentos de código, etc.).

In [28]:
X_train_T = X_train.T  # matriz término-documento: (tamaño del vocabulario) x (cantidad de documentos)
print(f"Shape matriz término-documento: {X_train_T.shape}")

# Palabras elegidas manualmente, representativas de distintos grupos temáticos del dataset
selected_words = ['car', 'god', 'hockey', 'medicine', 'computer']

for word in selected_words:
    word_idx = tfidfvect.vocabulary_[word]
    word_vec = X_train_T[word_idx]

    sim = cosine_similarity(word_vec, X_train_T)[0]
    mostsim = np.argsort(sim)[::-1][1:6]  # excluimos la propia palabra (similitud = 1)

    print("=" * 60)
    print(f"Palabra: '{word}'")
    print("Top 5 palabras más similares:")
    for rank, i in enumerate(mostsim, start=1):
        print(f"  {rank}. '{idx2word[i]}'  (similitud={sim[i]:.4f})")
    print()

Shape matriz término-documento: (101631, 11314)
Palabra: 'car'
Top 5 palabras más similares:
  1. 'cars'  (similitud=0.1797)
  2. 'criterium'  (similitud=0.1770)
  3. 'civic'  (similitud=0.1748)
  4. 'owner'  (similitud=0.1689)
  5. 'dealer'  (similitud=0.1681)

Palabra: 'god'
Top 5 palabras más similares:
  1. 'jesus'  (similitud=0.2688)
  2. 'bible'  (similitud=0.2616)
  3. 'that'  (similitud=0.2560)
  4. 'existence'  (similitud=0.2548)
  5. 'christ'  (similitud=0.2511)

Palabra: 'hockey'
Top 5 palabras más similares:
  1. 'ncaa'  (similitud=0.2743)
  2. 'nhl'  (similitud=0.2653)
  3. 'affiliates'  (similitud=0.2480)
  4. 'xenophobes'  (similitud=0.2426)
  5. 'sportschannel'  (similitud=0.2228)

Palabra: 'medicine'
Top 5 palabras más similares:
  1. 'strengthens'  (similitud=0.3654)
  2. 'dislikes'  (similitud=0.3464)
  3. 'nearer'  (similitud=0.3046)
  4. 'foremost'  (similitud=0.2836)
  5. 'surpress'  (similitud=0.2826)

Palabra: 'computer'
Top 5 palabras más similares:
  1. 'decwr

**Interpretación (Parte 4):**

Al transponer la matriz, dos palabras resultan "similares" cuando co-ocurren en
los mismos documentos con pesos TF-IDF parecidos; esto es una noción de similaridad
puramente distribucional, sin usar ningún modelo de lenguaje ni embeddings entrenados.

Los resultados fueron mixtos. Para 'car', 'god' y 'hockey' los vecinos resultaron
coherentes e interpretables (cars/dealer, jesus/bible, nhl/ncaa), confirmando la
hipótesis de que estas palabras capturan bien su dominio temático. Sin embargo,
'medicine' y 'computer' dieron vecinos poco interpretables (strengthens, dislikes,
decwriter, harkens), probablemente porque el vectorizador original no filtra
stopwords ni términos de baja frecuencia. Esto ilustra justamente lo que advierte
la consigna: elegir palabras a mano no garantiza vecinos interpretables si el
vectorizador de base no está optimizado.


## Conclusiones generales

- La similaridad coseno sobre vectores TF-IDF (Parte 1) captura relaciones léxicas razonables entre documentos, y
  en general es coherente con la etiqueta de clase, aunque no siempre.
- El modelo por prototipos / 1-NN (Parte 2) funciona como *baseline* pero es esperable que rinda claramente peor que
  Naïve Bayes, al depender de un único documento de referencia en vez de agregar estadística de toda la clase.
- Los modelos Naïve Bayes (Parte 3) lograron el mejor F1 macro (0.6954, ComplementNB
  con TFIDF default y alpha=0.1), muy por encima del modelo por prototipos (0.5050).
  El nivel de suavizado óptimo (alpha) resultó distinto según el modelo: bajar alpha
  ayudó de forma consistente a MultinomialNB, pero perjudicó a ComplementNB cuando fue
  demasiado bajo (0.01). El preprocesamiento adicional del vectorizador (stopwords,
  min_df/max_df, sublinear_tf) no mostró mejoras claras sobre TF-IDF por defecto, y
  en ningún caso hizo falta modificar ngram_range.
- Transponer la matriz documento-término (Parte 4) permite reutilizar exactamente la misma noción de similaridad
  coseno para comparar palabras en lugar de documentos, revelando agrupamientos temáticos coherentes con los 20
  grupos de noticias originales.